# Predicting CRM Sales Opportunities

This notebook is the CRM-first workflow for the CS 582 Group 2 project. It uses only information available when a deal is engaging.

In [ ]:
from pathlib import Path

repo = Path('/content/cs582-project')
if not repo.exists():
    !git clone https://github.com/thai-phan/cs582-project.git /content/cs582-project
%cd /content/cs582-project
!python -m pip install -q -r requirements.txt

## 1. Load and validate CRM data
Closed Won/Lost deals are labeled data. Open deals are kept separate.

In [ ]:
from src.datasets import load

dataset = load('crm')
dataset.report

In [ ]:
forbidden = {'deal_stage', 'close_date', 'close_value', 'opportunity_id'}
assert forbidden.isdisjoint(dataset.feature_columns)
print('Leakage check passed')
print('Training rows:', len(dataset.frame))
print('Scorable Engaging deals:', len(dataset.extra['scorable_open_deals']))

## 2. Train honest baseline models
The earliest 80% of deals are training data and the latest 20% are test data.

In [ ]:
from src.baselines import run

results = run(dataset, how='temporal')
display(results.round(4))

A ROC-AUC near 0.50 is not a code failure. It means the source data has little honest pre-close signal. Using `close_value` would leak the final outcome.

## 3. Score open Engaging deals
Random Forest is fitted again on the training period and used only as the first working demonstration. The final model will be selected after MLP and TabNet are evaluated.

In [ ]:
from pathlib import Path
from src.baselines import make_models, split
from src.crm_workflow import score_open_deals

X_train, X_test, y_train, y_test = split(dataset, 'temporal')
model = make_models(dataset)['random_forest']
model.fit(X_train, y_train)
open_predictions = score_open_deals(model, dataset)
output = Path('reports/crm/open_deal_predictions.csv')
output.parent.mkdir(parents=True, exist_ok=True)
open_predictions.to_csv(output, index=False)
display(open_predictions.head(10))
print('Saved:', output)